# AgriVision POC - banana disease classifier

Trains an 8-class banana disease classifier on the public
[PFSD-Musa](https://huggingface.co/datasets/Project-AgML/PFSD_Musa_banana_disease_classification)
dataset (CC-BY-4.0) and exposes it as a Streamlit page.

**Before you start:** `Runtime > Change runtime type > T4 GPU`. Training takes about
10 minutes on a T4 versus ~75 minutes on CPU.

Run the cells in order. Steps 3 and 7 are optional.


## 0 - Confirm the GPU


In [ ]:
import torch, os
print('torch     ', torch.__version__)
print('cuda      ', torch.cuda.is_available())
print('device    ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('cpu cores ', os.cpu_count())
if not torch.cuda.is_available():
    print('\n>> No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.')


## 1 - Upload the code

Run this cell and pick **`agrivision_poc.zip`** from the `poc/` folder on your machine.

It contains only the source files - no data, no weights, no virtualenv.


In [ ]:
from google.colab import files
import zipfile, os, shutil

if os.path.exists('/content/poc'):
    shutil.rmtree('/content/poc')

uploaded = files.upload()          # choose agrivision_poc.zip
name = next(iter(uploaded))
with zipfile.ZipFile(name) as z:
    z.extractall('/content')

%cd /content/poc
print()
!ls -1


### Alternative: clone from git

Once the code lives in a repo, replace step 1 with this and skip the upload entirely.


In [ ]:
# %cd /content
# !git clone https://github.com/<you>/<repo>.git
# %cd /content/<repo>/poc


## 2 - Install dependencies

Colab already ships a CUDA build of torch, torchvision, pandas, numpy, scikit-learn,
pyarrow and Pillow. **Do not `pip install torch`** - it would replace the GPU build
with a slower one and waste several minutes.

We also skip `uv sync` here: it would build a separate virtualenv that this notebook's
kernel cannot see. In Colab, plain `pip` into the running kernel is the right move.

So only two packages are actually missing.


In [ ]:
!pip install -q timm streamlit

import timm, streamlit
print('timm     ', timm.__version__)
print('streamlit', streamlit.__version__)


## 3 - Optional: keep weights on Google Drive

A Colab runtime is wiped when it disconnects, taking `artifacts/` with it. This points
`artifacts/` at Drive so a trained model survives a restart - worth it before a long
run, skippable if you just want a quick look.

**Re-run this cell every time you re-run step 1.** The upload cell deletes
`/content/poc` and the symlink with it, so your earlier artifacts stop being visible
until the link is recreated.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/agrivision_artifacts'
os.makedirs(dest, exist_ok=True)

link = '/content/poc/artifacts'
if os.path.islink(link):
    os.unlink(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
os.symlink(dest, link)

print('artifacts/ ->', os.path.realpath(link))


## 4 - Build the dataset

Downloads the parquet, writes `data/images/<class>/*.jpg` and `data/manifest.csv`.

Watch the group count in the output. The 6,700 images collapse into roughly **2,029
near-duplicate groups** - about 70% of the dataset is repeat frames of the same scene.
Training splits by group, never by image, so the test score is not a memory test.

Takes about a minute.


In [ ]:
!python prepare_data.py --hf-id Project-AgML/PFSD_Musa_banana_disease_classification --clean


## 5 - Train

GPU-appropriate settings: bigger batch, and `--workers 2` because a Colab VM only has
2 vCPUs - asking for more makes the data loader slower, not faster.

`--class-weight-power` controls how hard the loss compensates for class imbalance:
`0` = none, `1` = full inverse frequency, `0.5` = square root (the default, and what
you want here). Full inverse frequency gives `panama_disease` roughly 26x the weight of
`pseudostem_weevil` on this dataset, and the model responds by abandoning the dominant
class - weevil recall collapses to ~0.37 while three quarters of its confident
`panama_disease` calls are actually weevil. The square root keeps the rare classes
represented without inverting the prior.

`--version 0.2.0` writes a new artifact folder rather than overwriting an existing
`0.1.0`, so both stay selectable in the Streamlit sidebar and you can compare directly.

About 15 minutes on a T4 at 20 epochs.

In [ ]:
!python train.py \
    --epochs 20 \
    --freeze-epochs 2 \
    --batch-size 64 \
    --workers 2 \
    --image-size 224 \
    --class-weight-power 0.5 \
    --name banana-clf \
    --version 0.2.0

## 6 - Read the results

`macro-F1` is the headline, not accuracy: with `pseudostem_weevil` at 2,736 images and
`panama_disease` at 102, accuracy can look healthy while the rare classes are ignored.

`ECE` is calibration error - how much the printed confidence can be believed. It should
drop sharply after temperature scaling.

The single row to read first is **`pseudostem_weevil` recall**. It is 40% of the test
set, so macro-F1 moves with it more than with anything else, and a low value means the
weighting is pushing predictions off the dominant class.

In [ ]:
import json, glob, pandas as pd

art = sorted(glob.glob('artifacts/*'))[-1]
m = json.load(open(f'{art}/metrics.json'))

print(art)
print(f"macro-F1  {m['test_macro_f1']:.4f}")
print(f"accuracy  {m['test_accuracy']:.4f}")
print(f"ECE       {m['ece_before_calibration']:.4f} -> {m['ece_after_calibration']:.4f}"
      f"  (temperature {m['temperature']:.3f})")
print(f"epochs    {m.get('epochs', '?')}   class_weight_power {m.get('class_weight_power', '?')}")

print('\nper class')
display(pd.DataFrame(m['per_class']).T)

print('confusion matrix  (rows = truth, columns = prediction)')
display(pd.read_csv(f'{art}/confusion_matrix.csv', index_col=0))

### Comparing two runs

If you trained more than one version, this lines them up. Look at whether
`pseudostem_weevil` recall improved and what it cost the rare classes.

In [ ]:
import json, glob, pandas as pd

rows = {}
for a in sorted(glob.glob('artifacts/*')):
    try:
        m = json.load(open(f'{a}/metrics.json'))
    except FileNotFoundError:
        continue
    r = {'macro_f1': m['test_macro_f1'], 'accuracy': m['test_accuracy'],
         'ece': m['ece_after_calibration'], 'cw_power': m.get('class_weight_power'),
         'epochs': m.get('epochs')}
    r.update({f"recall:{k}": v['recall'] for k, v in m['per_class'].items()})
    rows[a.split('/')[-1]] = r

display(pd.DataFrame(rows).T if rows else 'no artifacts yet')

### Sanity check on a few images

In the confusion matrix above, find the cell where `black_sigatoka` meets
`yellow_sigatoka`. They are visually adjacent diseases with different treatments, so
that is the cell that matters most.


In [ ]:
!python predict.py data/images/black_sigatoka/*.jpg --top-k 3 2>/dev/null | head -24


## 7 - Optional: the Streamlit page

Colab cannot serve a port directly, so this opens a free Cloudflare tunnel - no signup,
no account.

**This cell keeps running on purpose.** Look for a line like
`https://something-random.trycloudflare.com` in the output and open it. Stop the cell
to shut the page down.


In [ ]:
import subprocess, os, time

if not os.path.exists('/content/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--server.address', '0.0.0.0', '--browser.gatherUsageStats', 'false'],
    cwd='/content/poc',
    stdout=open('/content/streamlit.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(10)
print('streamlit started; opening tunnel - look for the trycloudflare.com URL below\n')

!/content/cloudflared tunnel --url http://localhost:8501 --no-autoupdate


If the page errors, read the Streamlit log:


In [ ]:
!tail -25 /content/streamlit.log


## 8 - Swapping in the client's dataset later

Everything downstream reads only `data/manifest.csv`, so a new dataset means re-running
step 4 with a different source. `train.py`, `predict.py`, `app.py` and
`streamlit_app.py` need no edits - class names, class count and label order are all
derived from the manifest.

Raise `--image-size` to 384 if the client's photos are higher resolution than PFSD's
256 px; that is where early-stage Sigatoka detection lives, and this dataset cannot
support it.


In [ ]:
# a folder of class-named subdirectories, uploaded or mounted from Drive
# !python prepare_data.py --image-dir /content/drive/MyDrive/client_photos --clean
# !python train.py --epochs 20 --batch-size 64 --workers 2 --image-size 384 --version 1.0.0

# or a different HuggingFace dataset
# !python prepare_data.py --hf-id some-org/some-dataset --clean

---

### Before quoting any number from this notebook

- Measured on a held-out group split of **public** data, not field photos. Expect a
  drop on a phone photo of a real canopy.
- Source images are 256x256 and already downsampled, so fine lesion detail is partly
  gone. Early-stage detection is understated here and unvalidated at full resolution.
- There is **no `healthy` class and no `not_a_leaf` class**. The model assigns one of 8
  diseases to every image, including a photo of a shoe. Closing that gap is the first
  job for client data.
- `panama_disease` (102 images) and `banana_fruit_scarring_beetle` (150) fall below the
  200-per-class bar in the design doc. Panama / Fusarium TR4 is vascular and notifiable
  - treat its output as a placeholder, not a capability.

Attribution: Medhi & Deb, *PSFD-Musa*, Data in Brief 43:108427, 2022. CC-BY-4.0.
